Skeletonization: supervised, inference
======================================

In this notebook we use the supervised module to extract length and head width using a model trained on manually annotated data. We will use the script `skeletonization/main_supervised_skeletons_inference.py` to extract skeletons form the clips. 

First, we need to set up some running parameters for the script: 

In [4]:
from pathlib import Path
import yaml

cwd = Path.cwd()
ROOT_DIR = cwd.parent.absolute()

# Alternatively, change to your working directory:
# ROOT_DIR = Path("path/to/your/working/dir/here")

MODEL_S = "mit-b2-v0"

arguments = {
    "input_dir": ROOT_DIR / "data/mzb_example_data/derived/blobs/", 
    "input_type": "external", 
    "input_model": ROOT_DIR / f"models/mzb-skeleton-models/{MODEL_S}",
    "output_dir": ROOT_DIR / "results/mzb_example/skeletons/supervised_skeletons/", 
    "save_masks": ROOT_DIR / "data/mzb_example_data/derived/skeletons/supervised_skeletons/",
    "config_file": ROOT_DIR / "configs/mzb_example_config.yaml",
    "verbose": False
}
    
with open(str(arguments["config_file"]), "r") as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)

# cfg["trcl_gpu_ids"] = None
print(arguments)

{'input_dir': WindowsPath('d:/mzb-suite/data/mzb_example_data/derived/blobs'), 'input_type': 'external', 'input_model': WindowsPath('d:/mzb-suite/models/mzb-skeleton-models/mit-b2-v0'), 'output_dir': WindowsPath('d:/mzb-suite/results/mzb_example/skeletons/supervised_skeletons'), 'save_masks': WindowsPath('d:/mzb-suite/data/mzb_example_data/derived/skeletons/supervised_skeletons'), 'config_file': WindowsPath('d:/mzb-suite/configs/mzb_example_config.yaml'), 'verbose': False}


Convert to a dictionary for the scripts to parse. 

In [5]:
from mzbsuite.utils import cfg_to_arguments

# Transforms configurations dicts to argparse arguments
args = cfg_to_arguments(arguments)
cfg = cfg_to_arguments(cfg)
print(str(cfg))

{'glob_random_seed': 222, 'glob_root_folder': '/home/mzbuser/work/mzb-suite', 'glob_blobs_folder': '/home/mzbuser/work/mzb-suite/data/derived/blobs/', 'glob_local_format': 'pdf', 'model_logger': 'wandb', 'impa_image_format': 'jpg', 'impa_clip_areas': [2700, 4700, -1, -1], 'impa_area_threshold': 5000, 'impa_gaussian_blur': [21, 21], 'impa_gaussian_blur_passes': 3, 'impa_adaptive_threshold_block_size': 351, 'impa_mask_postprocess_kernel': [11, 11], 'impa_mask_postprocess_passes': 5, 'impa_bounding_box_buffer': 200, 'impa_save_clips_plus_features': True, 'lset_class_cut': 'order', 'lset_val_size': 0.1, 'trcl_learning_rate': 0.0001, 'trcl_batch_size': 8, 'trcl_weight_decay': 0, 'trcl_step_size_decay': 5, 'trcl_number_epochs': 75, 'trcl_save_topk': 1, 'trcl_num_classes': 8, 'trcl_model_pretrarch': 'convnext-small', 'trcl_num_workers': 16, 'trcl_wandb_project_name': 'mzb-classifiers', 'trcl_logger': 'wandb', 'trsk_learning_rate': 0.001, 'trsk_batch_size': 32, 'trsk_weight_decay': 0, 'trsk_st

We can load the code necessary to run the inference from the dedicated script, and call it with the arguments specified above. 

In [6]:
# from classification.main_classification_finetune import main as finetune_classifier
from scripts.skeletonization.main_supervised_skeleton_inference import main as skeletonization_supervised
?skeletonization_supervised

c:\Users\Pegoraro\AppData\Local\miniforge3\envs\mzbsuite\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Signature: skeletonization_supervised(args, cfg)
Docstring:
Function to run inference of skeletons (body, head) on macrozoobenthos images clips, using a trained model.

Parameters
----------
args : argparse.Namespace
    Namespace containing the arguments passed to the script. Notably:

        - input_dir: path to the directory containing the images to be classified
        - input_type: type of input data, either "val" or "external"
        - input_model: path to the directory containing the model to be used for inference
        - output_dir: path to the directory where the results will be saved
        - save_masks: path to the directory where the masks will be saved
        - config_file: path to the config file with train / inference parameters

cfg : dict
    Dictionary containing the configuration parameters.

Returns
-------
None. Saves the results in the specified folder.
File:      d:\mzb-suite\scripts\skeletonization\main_supervised_skeleton_inference.py
Type:      function

Now we can call the function and run the inference on the images. 

In [7]:
skeletonization_supervised(args, cfg)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\Pegoraro\AppData\Local\miniforge3\envs\mzbsuite\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Pegoraro\AppData\Local\miniforge3\envs\mzbsuite\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'predict_dataloader' to speed up the dataloader worker initialization.


Predicting DataLoader 0: 100%|██████████| 10/10 [00:09<00:00,  1.07it/s]


100%|██████████| 40/40 [01:16<00:00,  1.90s/it]


This produces a `.csv` file with the predictions for body length and head width (saved in `output_dir`), as well as the predicted skeletons for body length and head for each clip (saved in `save_masks`). 